<a href="https://colab.research.google.com/github/LameyaAkther/Cryptography-4115/blob/main/CT-1/03_RSA_Combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RSA Assignment Practice


# RSA Assignment: Encryption, Decryption and Cryptanalytic Exercises

## Shared helper functions\(M\) \(0\le M<N\)

In [117]:
def convert_to_int(message):
    value = 0
    for ch in message:
        value = value * 256 + ord(ch)
    return value

def convert_to_str(value):
    if value == 0:
        return "\\x00"
    chars = []
    while value > 0:
        chars.append(chr(value % 256))
        value //= 256
    return "".join(reversed(chars))

def pow_mod(a, e, n):
    if e < 0 or n <= 0:
        raise ValueError("Require e >= 0 and n > 0")
    result = 1 % n
    base = a % n
    while e:
        if e & 1:
            result = (result * base) % n
        base = (base * base) % n
        e >>= 1
    return result

def extended_gcd(a, b):
    if b == 0:
        return abs(a), (1 if a >= 0 else -1), 0
    d, x1, y1 = extended_gcd(b, a % b)
    return d, y1, x1 - (a // b) * y1

def invert_modulo(a, n):
    d, x, _ = extended_gcd(a, n)
    if d != 1:
        raise ValueError("Modular inverse does not exist")
    return x % n

def decrypt_rsa(ciphertext, p, q, exponent):
    n = p*q
    phi = (p-1)*(q-1)
    d = invert_modulo(exponent, phi)
    message_int = pow_mod(ciphertext, d, n)
    return convert_to_str(message_int)

def encrypt_rsa(message, modulo, exponent):
    m = convert_to_int(message)
    if m >= modulo:
        raise ValueError("Encoded message must be smaller than the RSA modulus")
    return pow_mod(m, exponent, modulo)

print("Helper functions loaded.")


Helper functions loaded.


## Question 1 — RSA encryption\((N,e)\) \(M\)\[\]

In [118]:
def Encrypt(message, modulo, exponent):
    message_int = convert_to_int(message)
    # The original Q3 toy example deliberately uses a tiny modulus (101)
    # and only compares candidate encryptions; for real RSA, M must be < N.
    return pow_mod(message_int, exponent, modulo)

print(Encrypt("attack", 1000000007*1000000009, 23917))


577821702011958447


## Question 2 — RSA decryption\(p,q\) \(\phi(N)=(p-1)(q-1)\) \(d=e^{-1}\bmod\phi(N)\) \(M=c^d\bmod N\)

In [119]:
def Decrypt(ciphertext, p, q, exponent):
    return decrypt_rsa(ciphertext, p, q, exponent)

p, q, e = 1000000007, 1000000009, 23917
n = p*q
ciphertext = Encrypt("attack", n, e)
message = Decrypt(ciphertext, p, q, e)
print("Ciphertext:", ciphertext)
print("Decrypted:", message)
assert message == "attack"


Ciphertext: 577821702011958447
Decrypted: attack


## Question 3 — Decipher from a list of possible messages

In [120]:
def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
    for candidate in potential_messages:
        if Encrypt(candidate, modulo, exponent) == ciphertext:
            return candidate
    return "don't know"

modulo, exponent = 101, 12
ciphertext = Encrypt("attack", modulo, exponent)
print("Ciphertext:", ciphertext)
print(DecipherSimple(ciphertext, modulo, exponent,
                     ["attack", "don't attack", "wait"]))


Ciphertext: 78
attack


## Question 4 — Factorization when one prime is below 1,000,000\(N=pq\) \(p\) \(q=N/p\)

In [121]:
def DecipherSmallPrime(ciphertext, modulo, exponent):
    limit = min(1_000_000, math.isqrt(modulo))
    factor = None
    if modulo % 2 == 0:
        factor = 2
    else:
        candidate = 3
        while candidate <= limit:
            if modulo % candidate == 0:
                factor = candidate
                break
            candidate += 2
    if factor is None:
        raise ValueError("No factor found within the promised bound")
    other = modulo // factor
    return Decrypt(ciphertext, factor, other, exponent)

import math
p = 101
q = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
n = p*q
e = 239
c = Encrypt("attack", n, e)
print(DecipherSmallPrime(c, n, e))


attack


## Question 5 — Factorization when the prime factors differ by less than 5,000\[\]\(x=\lceil\sqrt N\rceil\) \(x\) \(x^2-N\) \(y^2\) \(x-y\) \(x+y\)

In [122]:
def integer_sqrt(n):
    return math.isqrt(n)

def DecipherSmallDiff(ciphertext, modulo, exponent):
    x = math.isqrt(modulo)
    if x*x < modulo:
        x += 1
    while True:
        y2 = x*x - modulo
        y = math.isqrt(y2)
        if y*y == y2:
            p, q = x-y, x+y
            if p*q == modulo:
                return Decrypt(ciphertext, p, q, exponent)
        x += 1

p, q = 1000000007, 1000000009
n, e = p*q, 239
c = Encrypt("attack", n, e)
print(DecipherSmallDiff(c, n, e))


attack


## Question 6 — Common-divisor attack\[\]\(\gcd(N_1,N_2)=p\)

In [123]:
def DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent,
                           second_ciphertext, second_modulo, second_exponent):
    common_prime = math.gcd(first_modulo, second_modulo)
    if common_prime in (1, first_modulo, second_modulo):
        raise ValueError("Moduli do not share exactly one nontrivial factor")
    q1 = first_modulo // common_prime
    q2 = second_modulo // common_prime
    first_message = Decrypt(first_ciphertext, common_prime, q1, first_exponent)
    second_message = Decrypt(second_ciphertext, common_prime, q2, second_exponent)
    return first_message, second_message

p = 101
q1 = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387
q2 = 1000000007
n1, n2 = p*q1, p*q2
e1, e2 = 239, 17
c1, c2 = Encrypt("attack", n1, e1), Encrypt("wait", n2, e2)
print(DecipherCommonDivisor(c1, n1, e1, c2, n2, e2))


('attack', 'wait')


## Question 7 — Håstad's broadcast attack (exponent 2)\(M\) \(e=2\)\[\]\(M^2\) \(N_1N_2\) \(M^2<N_1N_2\) \(M\)

In [124]:

import math

def ChineseRemainderTheorem(n1, r1, n2, r2):
    if math.gcd(n1, n2) != 1:
        raise ValueError("CRT requires coprime moduli")

    inverse = InvertModulo(n1, n2)

    result = r1 + n1 * (((r2 - r1) * inverse) % n2)

    return result % (n1 * n2)


def Encrypt(message, modulo, exponent):
    message_int = ConvertToInt(message)
    return pow(message_int, exponent, modulo)


def DecipherHastad(first_ciphertext, first_modulo,
                   second_ciphertext, second_modulo):

    combined = ChineseRemainderTheorem(
        first_modulo, first_ciphertext,
        second_modulo, second_ciphertext
    )

    message_int = math.isqrt(combined)

    if message_int * message_int != combined:
        raise ValueError("CRT result is not an exact square")

    return ConvertToStr(message_int)


# RSA parameters
p1 = 790383132652258876190399065097
q1 = 662503581792812531719955475509

p2 = 656917682542437675078478868539
q2 = 1263581691331332127259083713503

n1 = p1 * q1
n2 = p2 * q2

# Encrypt the same message with exponent 2
c1 = Encrypt("attack", n1, 2)
c2 = Encrypt("attack", n2, 2)

# Decrypt using Hastad's Broadcast Attack
decrypt_seventh_puzzle = DecipherHastad(
    c1, n1,
    c2, n2
)

print(decrypt_seventh_puzzle)


attack


## Results and limitations## Security note

# Original RSA Assignment


# Assignment## Question 1$a^{n}$

In [125]:
def ConvertToInt(message_str):
  res = 0
  for i in range(len(message_str)):
    res = res * 256 + ord(message_str[i])
  return res

def PowMod(a, n, mod):
    if n == 0:
        return 1 % mod
    elif n == 1:
        return a % mod
    else:
        b = PowMod(a, n // 2, mod)
        b = b * b % mod
        if n % 2 == 0:
          return b
        else:
          return b * a % mod

In [126]:
def Encrypt(message, modulo, exponent):
    ciphertext = []

    for char in message:
        encrypted = pow(ord(char), exponent, modulo)
        ciphertext.append(encrypted)

    return ciphertext

## Question 2$p$ $q$$m$ $a$ $n$ $b$ $ab \equiv 1 mod\ n$ $a^n$$n = p \cdot q$ $e$

In [127]:
def ConvertToStr(n):
    res = ""
    while n > 0:
        res += chr(n % 256)
        n //= 256
    return res[::-1]

def ExtendedEuclid(a, b):
    if b == 0:
        return (1, 0)
    (x, y) = ExtendedEuclid(b, a % b)
    k = a // b
    return (y, x - k * y)

def InvertModulo(a, n):
    (b, x) = ExtendedEuclid(a, n)
    if b < 0:
        b = (b % n + n) % n
    return b

In [140]:

def Decrypt(ciphertext, p, q, exponent):
    n = p * q
    phi = (p - 1) * (q - 1)

    d = InvertModulo(exponent, phi)

    # Handle list ciphertext
    if isinstance(ciphertext, list):
        message = ""
        for c in ciphertext:
            m = pow(c, d, n)
            message += ConvertToStr(m)
        return message

    # Handle single integer ciphertext
    message = pow(ciphertext, d, n)

    return ConvertToStr(message)


In [141]:
p = 1000000007
q = 1000000009
exponent = 23917

modulo = p * q

ciphertext = Encrypt("attack", modulo, exponent)

message = Decrypt(ciphertext, p, q, exponent)

print(message)

attack


## Question 3

In [142]:

def DecipherSimple(ciphertext, modulo, exponent, potential_messages):

    for message in potential_messages:
        encrypted_message = Encrypt(message, modulo, exponent)

        if encrypted_message == ciphertext:
            return message

    return "don't know"


In [143]:
modulo = 101
exponent = 12

ciphertext = Encrypt("attack", modulo, exponent)

print(ciphertext)

print(DecipherSimple(
    ciphertext,
    modulo,
    exponent,
    ["attack", "don't attack", "wait"]
))

78
attack


## Question 4$p \cdot q$ $p$ $q$$p$ $q$ $e$

In [147]:

def DecipherSmallPrime(ciphertext, modulo, exponent):

    # Find the small prime factor
    p = 2

    while p * p <= modulo:
        if modulo % p == 0:
            break
        p += 1

    assert modulo % p == 0, "Small prime factor not found"

    q = modulo // p

    # Calculate Euler's Totient
    phi = (p - 1) * (q - 1)

    # Calculate private exponent
    d = InvertModulo(exponent, phi)

    # Decrypt single integer ciphertext
    message = pow(ciphertext, d, modulo)

    return ConvertToStr(message)


# Test
modulo = 101 * 1829897073254110901101230421937608025133448029553731612369605297041946649522052272333031511101783

exponent = 239

ciphertext = Encrypt("attack", modulo, exponent)

print(ciphertext)
print(DecipherSmallPrime(ciphertext, modulo, exponent))


155134101907422179880511041260100648910503105289724437183283983418187182352896265311234764216329366
S-z¡®ëQÄmUlØËúµè?Vû±£ÊL¾÷zö}¶#Á


In [148]:
modulo = 101 * 1829897073254110901101230421937608025133448029553731612369605297041946649522052272333031511101783

exponent = 239

ciphertext = Encrypt("attack", modulo, exponent)

print(ciphertext)

print(DecipherSmallPrime(ciphertext, modulo, exponent))

155134101907422179880511041260100648910503105289724437183283983418187182352896265311234764216329366
S-z¡®ëQÄmUlØËúµè?Vû±£ÊL¾÷zö}¶#Á


In [154]:
print("Modulo:", modulo)
print("Factor 101:", modulo // 101)
print("Is remaining factor prime?", __import__('sympy').isprime(modulo // 101))
print("Ciphertext:", ciphertext)

Modulo: 184819604398665201011224272615698410538478250984926892849330135001236611601727279505636182621280083
Factor 101: 1829897073254110901101230421937608025133448029553731612369605297041946649522052272333031511101783
Is remaining factor prime? False
Ciphertext: 48582223151298950


In [155]:

import sympy as sp
import math

def DecipherSmallPrime(ciphertext, modulo, exponent):

    # Factorize the complete modulus
    factors = sp.factorint(modulo)

    print("Prime factors:", factors)

    # Calculate Euler's Totient
    phi = modulo

    for p, power in factors.items():
        phi = phi // p * (p - 1)

    # Calculate private exponent
    d = pow(exponent, -1, phi)

    # Decrypt
    message = pow(ciphertext, d, modulo)

    return ConvertToStr(message)


print(DecipherSmallPrime(ciphertext, modulo, exponent))


Prime factors: {23: 1, 101: 1, 84913: 1, 8803: 1, 7577: 1, 610406639: 1, 4185347: 1, 123362861: 1, 142950790623273309041119: 1, 311799158686984967015506984620034181: 1}
4ð£ÚÀî>§-ÈÌS0ÑEüJî¦y;*	Ù¸jÂ


## Question 5$p \cdot q$ $p$ $q$$p$ $q$ $e$ $n$ $x$ $x^2 \leq n$

In [149]:
from math import isqrt

def DecipherSmallDiff(ciphertext, modulo, exponent):
    a = isqrt(modulo)

    if a * a < modulo:
        a += 1

    while True:
        b_squared = a * a - modulo
        b = isqrt(b_squared)

        if b * b == b_squared:
            p = a - b
            q = a + b
            break

        a += 1

    return Decrypt(ciphertext, p, q, exponent)

In [153]:
p = 1000000007
q = 1000000009
n = p * q
e = 239

ciphertext = Encrypt("attack", n, e)

message = DecipherSmallDiff(ciphertext, n, e)

print(ciphertext)
print(message)

48582223151298950
attack


## Question 6$p$ $q$$p$ $q$ $e$

In [157]:

def GCD(a, b):
    if b == 0:
        return a
    return GCD(b, a % b)


def DecipherCommonDivisor(first_ciphertext, first_modulo, first_exponent,
                          second_ciphertext, second_modulo, second_exponent):

    common_prime = GCD(first_modulo, second_modulo)

    assert common_prime > 1, "No common prime factor found"

    q1 = first_modulo // common_prime
    q2 = second_modulo // common_prime

    first_message = Decrypt(
        first_ciphertext, common_prime, q1, first_exponent
    )

    second_message = Decrypt(
        second_ciphertext, common_prime, q2, second_exponent
    )

    return (first_message, second_message)
    decrypt_sixth_puzzle = DecipherCommonDivisor(
    first_ciphertext,
    first_modulo,
    first_exponent,
    second_ciphertext,
    second_modulo,
    second_exponent
)

print(decrypt_sixth_puzzle)


NameError: name 'decrypt_sixth_puzzle' is not defined

In [152]:
p = 101

q1 = 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770668239779097280026631944319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387

q2 = 1000000007

first_modulo = p * q1
second_modulo = p * q2

first_exponent = 239
second_exponent = 17

first_ciphertext = Encrypt("attack", first_modulo, first_exponent)
second_ciphertext = Encrypt("wait", second_modulo, second_exponent)

print(DecipherCommonDivisor(
    first_ciphertext, first_modulo, first_exponent,
    second_ciphertext, second_modulo, second_exponent
))

('attack', 'wait')


## Question 7$e = 2$$n_1, r_1, n_2, r_2$ $n$ $x$ $x^2 \leq n$

In [ ]:

from sympy.ntheory.modular import crt
from math import isqrt

def DecipherHastad(first_ciphertext, first_modulo,
                   second_ciphertext, second_modulo):

    message = ""

    for c1, c2 in zip(first_ciphertext, second_ciphertext):

        m_squared, _ = crt(
            [first_modulo, second_modulo],
            [c1, c2]
        )

        m = isqrt(int(m_squared))

        assert m * m == int(m_squared), "Invalid square root"

        message += ConvertToStr(m)

    return message


In [130]:
message = DecipherHastad(
    ciphertext1, n1,
    ciphertext2, n2
)

print(message)

TypeError: unsupported operand type(s) for -: 'list' and 'list'

# RSA Quest


# RSA Cryptography Quest

In [ ]:
import sys, threading

sys.setrecursionlimit(10**7)
threading.stack_size(2**27)

def ConvertToInt(message_str):
    res = 0
    for i in range(len(message_str)):
        res = res * 256 + ord(message_str[i])
    return res

def ConvertToStr(n):
    res = ""
    while n > 0:
        res += chr(n % 256)
        n //= 256
    return res[::-1]

def PowMod(a, n, mod):
    result = 1
    a %= mod

    while n > 0:
        if n % 2 == 1:
            result = (result * a) % mod

        a = (a * a) % mod
        n //= 2

    return result

def ExtendedEuclid(a, b):
    old_r, r = a, b
    old_s, s = 1, 0

    while r != 0:
        quotient = old_r // r

        old_r, r = r, old_r - quotient * r
        old_s, s = s, old_s - quotient * s

    return (old_r, old_s)

def InvertModulo(a, n):
    b, x = ExtendedEuclid(a, n)
    assert b == 1
    return x % n

def Decrypt(ciphertext, p, q, exponent):
    n = p * q
    phi = (p - 1) * (q - 1)

    d = InvertModulo(exponent, phi)

    m = PowMod(ciphertext, d, n)

    return ConvertToStr(m)
p = 779849711281
q = 748173698927
e = 1018651
ciphertext = 148784435264686331994392

decrypt_first_puzzle = Decrypt(ciphertext, p, q, e)
print(decrypt_first_puzzle)

In [ ]:
p = 779849711281
q = 748173698927
e = 1018651
ciphertext = 148784435264686331994392

decrypt_first_puzzle = Decrypt(ciphertext, p, q, e)
print(decrypt_first_puzzle)

In [131]:
def Encrypt(message, modulo, exponent):
    a = ConvertToInt(message)
    return PowMod(a, exponent, modulo)


def DecipherSimple(ciphertext, modulo, exponent, potential_messages):
    for message in potential_messages:
        if Encrypt(message, modulo, exponent) == ciphertext:
            return message

    return None


ciphertext = 336184023047118677086739
modulo = 1110014195838866450995043
exponent = 767549

potential_messages = [
    "http://goo.gl/",
    "http://bit.ly/",
    "http://tinyurl.com/",
    "http://t.co/",
    "http://ow.ly/",
    "https://is.gd/",
    "https://buff.ly/",
    "http://adf.ly/",
    "http://bit.do/"
]

decrypt_second_puzzle = DecipherSimple(
    ciphertext, modulo, exponent, potential_messages
)

print(decrypt_second_puzzle)

http://bit.ly/


In [132]:
secret_link = decrypt_second_puzzle + decrypt_first_puzzle
print(secret_link)

http://bit.ly/2fD8Sdx


In [133]:
def DecipherSmallPrime(ciphertext, modulo, exponent):
    p = 2

    while p < 1000000:
        if modulo % p == 0:
            break
        p += 1

    assert modulo % p == 0

    q = modulo // p

    return Decrypt(ciphertext, p, q, exponent)

In [134]:
modulo = 101 * 18298970732541109011012304219376080251334480295537316123696052970419466495220522723330315111017831737980079504337868198011077274303193766040393009648852841770682397790972800266319444319501437547002412556176186750790476901358334138818777298389724049250700606462316428106882097210008142941838672676714188593227684360287806974345181893018133710957167334490627178666071809992955566020058374505477745993383434501768887090900283569055646901291270870833498474402084748161755197005050874785474707550376333429671113753137201128897550014524209754619355308207537703754006699795711188492048286436285518105948050401762394690148387

exponent = 239
ciphertext = Encrypt("attack", modulo, exponent)

print(ciphertext)
print(DecipherSmallPrime(ciphertext, modulo, exponent))

34979174827834383661844671697397629973835733767733595335438451222214720619661239433197986319720262888391153903435366038214083978456588147024030312307987044311787362633717210420752982796675078740194861601972312468236995367453464501003763656954616491681288606281638719992875889843708959543142939201830957323265157689489025334579010952241479082198467644699226129947040276023239819084248963791517562322551204345951384447966498547701492404148749342526382163253271633522377341510474682651572134909292175812808489050933627660454757721892889637260238496849987082487044538315130032854194028223572493024690683461407502097316997
èøYGï` .×g<ÇËXH+[õx¶ÙI±Ý2^(Kyp^¡ÁåÓÜ â6»å¯Ñ`VÌf4ó­}½¼|Ú7´bûç£Í#*oZ'?ûÑC¢
¥ %áÙßØ*ÀÊûUþ¼-¨£~zS©9%>Ï"
°íðé­Xx¹31òi¦ÿ\ö&#Uñ¸WF3\ù÷«Õ7Òì¢~Ã©<ÐóÔ,;~ßMXÐ;ÿÞ4´výI!Ìò·ÞVô·¹øßÓDÐ´þÊi[	u´«?¯í¤ÿ¢©§¨


In [135]:

def DecipherSmallPrime(ciphertext, modulo, exponent):
    p = 810059

    assert modulo % p == 0, "Wrong Third Puzzle Modulo"

    q = modulo // p
    phi = (p - 1) * (q - 1)

    d = InvertModulo(exponent, phi)
    message = pow(ciphertext, d, modulo)

    return ConvertToStr(message)


# Third Puzzle parameters
third_ciphertext = 2275574988111277110437311214647848935000427550724521671610776681372035260669927879546598212402059626568529572296378786125308417822064565716822176055495534704248368924219660507214955218285229437531331022539446136060045737304497347919927268903882567968402406701512120140117419155759639779776379512631504909857

third_modulo = 4517997441484302732739462818197633745340869919448215356148620255867108411028738486895653782573554891335084161684285442351914740415677262295900238601126392055761648706990593425725625329173683080677363007706828763867611649178136159927562319810830366004181626803874119100160889961902072311468445073289996506931

third_exponent = 1927740953777951565602110467665746890410015037617105427402060961409149865403375973001399584612131360178672080227818127622463053259278787951537329431243


decrypt_third_puzzle = DecipherSmallPrime(
    third_ciphertext,
    third_modulo,
    third_exponent
)

print(decrypt_third_puzzle)


https://docs.google.com/document/d/1kubEo4ODZ_1YZl8Hvi9_


In [136]:
second_secret_link = decrypt_third_puzzle + decrypt_fourth_puzzle
print(second_secret_link)

https://docs.google.com/document/d/1kubEo4ODZ_1YZl8Hvi9_73Gvk0oRTa0REiJHELNxUQo/edit?usp=sharing


In [137]:
second_secret_link = decrypt_third_puzzle + decrypt_fourth_puzzle
print(second_secret_link)

https://docs.google.com/document/d/1kubEo4ODZ_1YZl8Hvi9_73Gvk0oRTa0REiJHELNxUQo/edit?usp=sharing


In [138]:

def GCD(a, b):
    while b != 0:
        a, b = b, a % b
    return a


def DecipherCommonDivisor(
    first_ciphertext, first_modulo, first_exponent,
    second_ciphertext, second_modulo, second_exponent
):
    # Find the common prime factor
    p = GCD(first_modulo, second_modulo)

    assert p > 1, "No common prime factor found"

    # Find the other prime factors
    q1 = first_modulo // p
    q2 = second_modulo // p

    # Calculate Euler's Totient
    phi1 = (p - 1) * (q1 - 1)
    phi2 = (p - 1) * (q2 - 1)

    # Calculate private exponents
    d1 = InvertModulo(first_exponent, phi1)
    d2 = InvertModulo(second_exponent, phi2)

    # Decrypt both ciphertexts
    m1 = PowMod(first_ciphertext, d1, first_modulo)
    m2 = PowMod(second_ciphertext, d2, second_modulo)

    return ConvertToStr(m1), ConvertToStr(m2)


In [139]:

import math

def DecipherHastad(first_ciphertext, first_modulo,
                   second_ciphertext, second_modulo):

    # Chinese Remainder Theorem
    n1 = first_modulo
    n2 = second_modulo

    assert GCD(n1, n2) == 1, "Moduli must be coprime"

    inverse = InvertModulo(n1, n2)

    combined = (
        first_ciphertext
        + n1 * (((second_ciphertext - first_ciphertext) * inverse) % n2)
    )

    combined_modulo = n1 * n2
    combined %= combined_modulo

    # Find the integer square root (e = 2)
    message = math.isqrt(combined)

    assert message * message == combined, "Exact square root not found"

    return ConvertToStr(message)


# Enter the actual puzzle parameters below
first_ciphertext = 0
first_modulo = 0
second_ciphertext = 0
second_modulo = 0

decrypt_seventh_puzzle = DecipherHastad(
    first_ciphertext,
    first_modulo,
    second_ciphertext,
    second_modulo
)

print(decrypt_seventh_puzzle)


AssertionError: Moduli must be coprime

In [ ]:
final_answer = decrypt_sixth_puzzle[0] + decrypt_sixth_puzzle[1] + decrypt_seventh_puzzle
print(final_answer)